In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Check current directory files
import os
print(os.listdir('.'))

# Load cats.csv
df = pd.read_csv('cats.csv')
print(df.head())
print(df.info())

['cats.csv', 'lab1.ipynb']
   48076  26989072  \
0  48077  26979449   
1  48078  26974960   
2  48079  26973986   
3  48080  26972127   
4  48081  26961377   

  https://www.petfinder.com/cat/jupiter-26989072/qc/montreal/chatopia-formerly-animal-adoption-montreal-qc35/?referrer_id=3830981a-ad4c-4f85-872c-ca9edc50a67e  \
0  https://www.petfinder.com/cat/elliot-26979449/...                                                                                                             
1  https://www.petfinder.com/cat/zarro-26974960/m...                                                                                                             
2  https://www.petfinder.com/cat/snickers-2697398...                                                                                                             
3  https://www.petfinder.com/cat/ms-bigglesworth-...                                                                                                             
4  https://www.petfinder.com/c

In [2]:
df = pd.read_csv('cats.csv', header=None)
print("Shape:", df.shape)
print(df.head(10))
print("\nUnique values per column:")
for col in df.columns:
    print(f"Col {col}: {df[col].nunique()} unique, sample: {df[col].unique()[:5]}")

Shape: (298, 11)
      0         1                                                  2    3   \
0  48076  26989072  https://www.petfinder.com/cat/jupiter-26989072...  Cat   
1  48077  26979449  https://www.petfinder.com/cat/elliot-26979449/...  Cat   
2  48078  26974960  https://www.petfinder.com/cat/zarro-26974960/m...  Cat   
3  48079  26973986  https://www.petfinder.com/cat/snickers-2697398...  Cat   
4  48080  26972127  https://www.petfinder.com/cat/ms-bigglesworth-...  Cat   
5  48081  26961377  https://www.petfinder.com/cat/midnight-2696137...  Cat   
6  48082  26958952  https://www.petfinder.com/cat/jinxy-26958952/n...  Cat   
7  48083  26948459  https://www.petfinder.com/cat/blackie-26948459...  Cat   
8  48084  26946316  https://www.petfinder.com/cat/princess-2694631...  Cat   
9  48085  26943348  https://www.petfinder.com/cat/pisces-26943348/...  Cat   

       4       5       6       7        8   \
0   Adult    Male  Medium     NaN  Persian   
1   Adult    Male  Medium    Lon

In [3]:
print("Breed distribution (Col 8):")
print(df[8].value_counts())
print("\nCoat length distribution (Col 7):")
print(df[7].value_counts(dropna=False))
print("\nAge distribution (Col 4):")
print(df[4].value_counts(dropna=False))
print("\nGender distribution (Col 5):")
print(df[5].value_counts(dropna=False))
print("\nSize distribution (Col 6):")
print(df[6].value_counts(dropna=False))

Breed distribution (Col 8):
8
Persian       149
Pixiebob      111
Ragamuffin     38
Name: count, dtype: int64

Coat length distribution (Col 7):
7
Long      115
NaN        96
Short      56
Medium     31
Name: count, dtype: int64

Age distribution (Col 4):
4
Adult     185
Young      60
Senior     33
Baby       20
Name: count, dtype: int64

Gender distribution (Col 5):
5
Female    161
Male      137
Name: count, dtype: int64

Size distribution (Col 6):
6
Medium         154
Large           83
Small           49
Extra Large     12
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load and clean dataset
data = df[[4, 5, 6, 7, 8]].copy()
data.columns = ['Age', 'Gender', 'Size', 'Coat', 'Breed']

# Handle missing values in Coat
data['Coat'] = data['Coat'].fillna('Unknown')

print(data.head())
print(data.isnull().sum())

# Ordinal mapping or One-Hot Encoding
# Let's map ordinal features where appropriate, or use One-Hot
age_map = {'Baby': 0, 'Young': 1, 'Adult': 2, 'Senior': 3}
gender_map = {'Female': 0, 'Male': 1}
size_map = {'Small': 0, 'Medium': 1, 'Large': 2, 'Extra Large': 3}
coat_map = {'Unknown': 0, 'Short': 1, 'Medium': 2, 'Long': 3}

data_encoded = data.copy()
data_encoded['Age'] = data['Age'].map(age_map)
data_encoded['Gender'] = data['Gender'].map(gender_map)
data_encoded['Size'] = data['Size'].map(size_map)
data_encoded['Coat'] = data['Coat'].map(coat_map)

X = data_encoded[['Age', 'Gender', 'Size', 'Coat']]
y = data_encoded['Breed']

# Train Test Split (e.g., 80/20 with random_state=42 and stratify=y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

k_values = [1, 3, 5, 7, 9, 11]
results = {}

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[k] = acc
    print(f"k = {k}: Accuracy = {acc:.4f}")

      Age  Gender    Size     Coat    Breed
0   Adult    Male  Medium  Unknown  Persian
1   Adult    Male  Medium     Long  Persian
2   Adult    Male   Small   Medium  Persian
3   Adult  Female   Small  Unknown  Persian
4  Senior  Female   Small     Long  Persian
Age       0
Gender    0
Size      0
Coat      0
Breed     0
dtype: int64
k = 1: Accuracy = 0.7667
k = 3: Accuracy = 0.7833
k = 5: Accuracy = 0.7833
k = 7: Accuracy = 0.7500
k = 9: Accuracy = 0.7833
k = 11: Accuracy = 0.7667


In [5]:
from sklearn.model_selection import cross_val_score

# Try One-Hot Encoding
X_ohe = pd.get_dummies(data[['Age', 'Gender', 'Size', 'Coat']], drop_first=True)
X_tr_ohe, X_te_ohe, y_tr, y_te = train_test_split(X_ohe, y, test_size=0.2, random_state=42, stratify=y)

scaler_ohe = StandardScaler()
X_tr_ohe_scaled = scaler_ohe.fit_transform(X_tr_ohe)
X_te_ohe_scaled = scaler_ohe.transform(X_te_ohe)

print("--- One-Hot Encoded ---")
for k in [1, 3, 5, 7, 9, 11]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr_ohe_scaled, y_tr)
    acc = accuracy_score(y_te, knn.predict(X_te_ohe_scaled))
    print(f"k = {k}: Accuracy = {acc:.4f}")

print("\n--- Cross-Validation (5-Fold) on Ordinal Scaled ---")
X_scaled = scaler.fit_transform(X)
for k in [1, 3, 5, 7, 9, 11]:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_scaled, y, cv=5)
    print(f"k = {k}: Mean CV Accuracy = {scores.mean():.4f}")

--- One-Hot Encoded ---
k = 1: Accuracy = 0.7167
k = 3: Accuracy = 0.8000
k = 5: Accuracy = 0.7667
k = 7: Accuracy = 0.7167
k = 9: Accuracy = 0.7667
k = 11: Accuracy = 0.7833

--- Cross-Validation (5-Fold) on Ordinal Scaled ---
k = 1: Mean CV Accuracy = 0.6845
k = 3: Mean CV Accuracy = 0.7079
k = 5: Mean CV Accuracy = 0.7013
k = 7: Mean CV Accuracy = 0.6946
k = 9: Mean CV Accuracy = 0.6982
k = 11: Mean CV Accuracy = 0.6947


In [6]:
print("Dataset Overview:")
print("Total rows:", len(data))
print("Target classes (Breed):")
print(data['Breed'].value_counts())
print("\nFeature Summary:")
for col in ['Age', 'Gender', 'Size', 'Coat']:
    print(f"\n{col}:")
    print(data[col].value_counts())

# Detailed Evaluation for k = 3, 5, 7 on Ordinal Encoded Data
knn3 = KNeighborsClassifier(n_neighbors=3).fit(X_train_scaled, y_train)
knn5 = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)
knn7 = KNeighborsClassifier(n_neighbors=7).fit(X_train_scaled, y_train)

y_pred3 = knn3.predict(X_test_scaled)
y_pred5 = knn5.predict(X_test_scaled)
y_pred7 = knn7.predict(X_test_scaled)

print("\nAccuracy Scores (Holdout Test Set 80/20):")
print(f"k = 3: {accuracy_score(y_test, y_pred3)*100:.2f}%")
print(f"k = 5: {accuracy_score(y_test, y_pred5)*100:.2f}%")
print(f"k = 7: {accuracy_score(y_test, y_pred7)*100:.2f}%")

print("\nClassification Report for Best k (k=3):")
print(classification_report(y_test, y_pred3))

Dataset Overview:
Total rows: 298
Target classes (Breed):
Breed
Persian       149
Pixiebob      111
Ragamuffin     38
Name: count, dtype: int64

Feature Summary:

Age:
Age
Adult     185
Young      60
Senior     33
Baby       20
Name: count, dtype: int64

Gender:
Gender
Female    161
Male      137
Name: count, dtype: int64

Size:
Size
Medium         154
Large           83
Small           49
Extra Large     12
Name: count, dtype: int64

Coat:
Coat
Long       115
Unknown     96
Short       56
Medium      31
Name: count, dtype: int64

Accuracy Scores (Holdout Test Set 80/20):
k = 3: 78.33%
k = 5: 78.33%
k = 7: 75.00%

Classification Report for Best k (k=3):
              precision    recall  f1-score   support

     Persian       0.74      0.97      0.84        30
    Pixiebob       0.89      0.77      0.83        22
  Ragamuffin       0.50      0.12      0.20         8

    accuracy                           0.78        60
   macro avg       0.71      0.62      0.62        60
weighted avg